# Ablation Study: LoRA + Augmentation
## Vietnamese-English Code-Switching ASR

**Methodology:**
- Ablation study → **val set only** (no test leakage)
- Best model selected from val → **holdout evaluated once**

| Condition | Speed Aug | SpecAugment | LoRA |
|---|---|---|---|
| Baseline | ❌ | ❌ | ❌ |
| + Aug only | ✅ | ✅ | ✅ |
| + LoRA only | ❌ | ❌ | ✅ |
| + Aug + LoRA | ✅ | ✅ | ✅ |

## 1. Imports

In [ ]:
import os, gc, re, json
import numpy as np
import pandas as pd
import torch
import librosa
import torchaudio.transforms as T
from dataclasses import dataclass, field
from typing import Any
from tqdm.notebook import tqdm
import datasets
from datasets import Dataset, load_from_disk
import evaluate
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, PeftModelForSeq2SeqLM

# CVE-2025-32434 patch — allow loading .bin with torch < 2.6
import transformers.modeling_utils as _mu
import torch as _torch
import safetensors.torch as _st

def _patched_load_state_dict(checkpoint_file, map_location=None, weights_only=False):
    if checkpoint_file.endswith(".safetensors"):
        return _st.load_file(checkpoint_file, device=map_location or "cpu")
    return _torch.load(checkpoint_file, map_location=map_location or "cpu", weights_only=False)

_mu.load_state_dict = _patched_load_state_dict

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("CVE patch applied.")

## 2. Config

In [ ]:
DATA_ROOT        = "."
TRAIN_AUDIO_DIR  = os.path.join(DATA_ROOT, "train", "audio")
TRAIN_TRANSCRIPT = os.path.join(DATA_ROOT, "train", "transcript.json")
DEV_AUDIO_DIR    = os.path.join(DATA_ROOT, "dev",   "audio")
DEV_TRANSCRIPT   = os.path.join(DATA_ROOT, "dev",   "transcript.json")
TEST_AUDIO_DIR   = os.path.join(DATA_ROOT, "test",  "audio")
TEST_TRANSCRIPT  = os.path.join(DATA_ROOT, "test",  "transcript.json")

MODEL_CONFIGS = {
    "whisper-large-v3": {
        "model_id":   "openai/whisper-large-v3",
        "language":   "vi",
        "task":       "transcribe",
        "output_dir": "./output/ablation/whisper-large-v3",
        "cache_dir":  "./cache/ablation/whisper-large-v3",
    },
    "PhoWhisper-large": {
        "model_id":   "vinai/PhoWhisper-large",
        "language":   "vi",
        "task":       "transcribe",
        "output_dir": "./output/ablation/PhoWhisper-large",
        "cache_dir":  "./cache/ablation/PhoWhisper-large",
    },
}

LORA_CONFIG = {
    "r":              32,
    "lora_alpha":     64,
    "lora_dropout":   0.1,
    "target_modules": ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    "bias":           "none",
}

TRAINING_CONFIG = {
    "num_train_epochs":            5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size":  8,
    "gradient_accumulation_steps": 2,
    "learning_rate":               5e-5,
    "weight_decay":                0.01,
    "warmup_steps":                500,
    "fp16":                        False,
    "bf16":                        True,
    "evaluation_strategy":         "epoch",
    "save_strategy":               "epoch",
    "load_best_model_at_end":      True,
    "metric_for_best_model":       "wer",
    "greater_is_better":           False,
    "early_stopping_patience":     3,
    "predict_with_generate":       True,
    "generation_max_length":       225,
    "logging_steps":               25,
    "dataloader_num_workers":      4,
    "gradient_checkpointing":      False,
}

AUG_CONFIG = {
    "speed_rates":     [0.9],
    "freq_mask_param": 27,
    "time_mask_param": 100,
    "n_freq_masks":    2,
    "n_time_masks":    2,
}

SAMPLING_RATE     = 16000
MAX_AUDIO_LEN_SEC = 30
BATCH_SIZE        = 8

os.environ["WANDB_DISABLED"]          = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
datasets.enable_progress_bar()
print("Config loaded.")

## 3. Data, Augmentation & Utilities

In [ ]:
# ── Audio & text ──────────────────────────────────────────────
def load_audio(path):
    audio, _ = librosa.load(path, sr=SAMPLING_RATE, mono=True)
    max_len = MAX_AUDIO_LEN_SEC * SAMPLING_RATE
    return audio[:max_len] if len(audio) > max_len else audio

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\[(vi|en)\]', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_samples(audio_dir, transcript_path):
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcripts = json.load(f)
    samples, missing = [], 0
    for filename, meta in transcripts.items():
        path = os.path.join(audio_dir, filename)
        if os.path.exists(path):
            samples.append({"audio_path": path, "filename": filename, "text": meta["text"]})
        else:
            missing += 1
    print(f"  Loaded {len(samples)} (skipped {missing} missing)")
    return samples

# ── Augmentation ───────────────────────────────────────────────
def apply_speed_perturbation(samples, speed_rates=None):
    if speed_rates is None:
        speed_rates = AUG_CONFIG["speed_rates"]
    augmented = []
    for sample in tqdm(samples, desc="Speed perturbation"):
        augmented.append(sample)
        audio = load_audio(sample["audio_path"])
        for rate in speed_rates:
            stretched = librosa.effects.time_stretch(audio, rate=rate)
            stretched = stretched[:MAX_AUDIO_LEN_SEC * SAMPLING_RATE]
            perturbed = sample.copy()
            perturbed["audio_array"] = stretched.astype(np.float32)
            perturbed["speed_rate"]  = rate
            augmented.append(perturbed)
    print(f"  {len(samples)} → {len(augmented)} samples ({len(augmented)//len(samples)}x)")
    return augmented

class SpecAugment:
    def __init__(self):
        self.freq_masking = T.FrequencyMasking(freq_mask_param=AUG_CONFIG["freq_mask_param"])
        self.time_masking = T.TimeMasking(time_mask_param=AUG_CONFIG["time_mask_param"])
        self.n_freq = AUG_CONFIG["n_freq_masks"]
        self.n_time = AUG_CONFIG["n_time_masks"]
    def __call__(self, x):
        x = x.clone()
        for _ in range(self.n_freq): x = self.freq_masking(x)
        for _ in range(self.n_time): x = self.time_masking(x)
        return x

# ── HF Dataset builders ────────────────────────────────────────
def _build_hf_dataset(samples, processor, has_audio_array=False):
    def preprocess(batch):
        audio = np.array(batch["audio_array"], dtype=np.float32) if has_audio_array and batch.get("audio_array") else load_audio(batch["audio_path"])
        batch["input_features"] = processor.feature_extractor(audio, sampling_rate=SAMPLING_RATE).input_features[0]
        batch["labels"]         = processor.tokenizer(batch["text"]).input_ids
        return batch
    ds = Dataset.from_list(samples).map(preprocess, num_proc=1)
    return ds.remove_columns([c for c in ds.column_names if c not in ["input_features", "labels"]])

def prepare_hf_dataset(samples, processor, language, task):
    processor.tokenizer.set_prefix_tokens(language=language, task=task)
    return _build_hf_dataset(samples, processor, has_audio_array=False)

def prepare_hf_dataset_aug(samples, processor, language, task):
    processor.tokenizer.set_prefix_tokens(language=language, task=task)
    return _build_hf_dataset(samples, processor, has_audio_array=True)

# ── Metrics ────────────────────────────────────────────────────
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_ser(references, hypotheses):
    return sum(r.strip() != h.strip() for r, h in zip(references, hypotheses)) / len(references) * 100

def compute_all_metrics(references, hypotheses):
    refs_n = [normalize_text(r) for r in references]
    hyps_n = [normalize_text(h) for h in hypotheses]
    return {
        "wer_raw":  round(wer_metric.compute(references=references, predictions=hypotheses) * 100, 2),
        "cer_raw":  round(cer_metric.compute(references=references, predictions=hypotheses) * 100, 2),
        "ser_raw":  round(compute_ser(references, hypotheses), 2),
        "wer_norm": round(wer_metric.compute(references=refs_n,   predictions=hyps_n)       * 100, 2),
        "cer_norm": round(cer_metric.compute(references=refs_n,   predictions=hyps_n)       * 100, 2),
        "ser_norm": round(compute_ser(refs_n, hyps_n), 2),
    }

print("Utilities defined.")

## 4. Model Builder & Train Function

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor:              Any
    decoder_start_token_id: int
    training:               bool = True
    spec_augment:           Any  = field(default=None)
    def __post_init__(self):
        if self.spec_augment is None:
            self.spec_augment = SpecAugment()
    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        if self.training:
            batch["input_features"] = self.spec_augment(batch["input_features"])
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch

def build_lora_model(model_id):
    print(f"Loading model: {model_id}")
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float32
    ).to("cuda")
    model.config.use_cache                     = False
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens    = []

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_CONFIG["r"], lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        target_modules=LORA_CONFIG["target_modules"],
        bias=LORA_CONFIG["bias"],
    )
    model = get_peft_model(model, lora_cfg)
    model.enable_input_require_grads()
    model.base_model.model.model.encoder.gradient_checkpointing = False
    model.print_trainable_parameters()

    original_forward = PeftModelForSeq2SeqLM.forward
    def patched_forward(self, input_ids=None, attention_mask=None, inputs_embeds=None,
                        decoder_input_ids=None, decoder_attention_mask=None,
                        decoder_inputs_embeds=None, labels=None, output_attentions=None,
                        output_hidden_states=None, return_dict=None, task_ids=None, **kwargs):
        if "input_features" in kwargs:
            input_features = kwargs.pop("input_features")
            with self._enable_peft_forward_hooks(**kwargs):
                kwargs = {k: v for k, v in kwargs.items() if k not in self.special_peft_forward_args}
                return self.base_model(
                    input_features=input_features, attention_mask=attention_mask,
                    decoder_input_ids=decoder_input_ids, decoder_attention_mask=decoder_attention_mask,
                    decoder_inputs_embeds=decoder_inputs_embeds, labels=labels,
                    output_attentions=output_attentions, output_hidden_states=output_hidden_states,
                    return_dict=return_dict, **kwargs,
                )
        return original_forward(
            self, input_ids=input_ids, attention_mask=attention_mask,
            inputs_embeds=inputs_embeds, decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask, decoder_inputs_embeds=decoder_inputs_embeds,
            labels=labels, output_attentions=output_attentions, output_hidden_states=output_hidden_states,
            return_dict=return_dict, task_ids=task_ids, **kwargs,
        )
    PeftModelForSeq2SeqLM.forward = patched_forward
    return model, processor

def _auto_resume(output_dir):
    if not os.path.exists(output_dir): return None
    ckpts = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not ckpts: return None
    latest = sorted(ckpts, key=lambda x: int(x.split("-")[-1]))[-1]
    print(f"  Auto-resuming from: {latest}")
    return os.path.join(output_dir, latest)

def train_condition(tag, model_id, train_dataset, dev_dataset,
                    language, task, output_dir, use_spec_augment=True):
    os.makedirs(output_dir, exist_ok=True)
    model, processor = build_lora_model(model_id)

    train_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
        training=use_spec_augment,
    )

    def compute_metrics(pred):
        pred_ids, label_ids = pred.predictions, pred.label_ids
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        return {"wer": round(wer_metric.compute(references=label_str, predictions=pred_str) * 100, 2)}

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
        gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        warmup_steps=TRAINING_CONFIG["warmup_steps"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        optim="adamw_torch",
        lr_scheduler_type="cosine_with_restarts",
        lr_scheduler_kwargs={"num_cycles": 2},
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
        fp16=TRAINING_CONFIG["fp16"],
        bf16=TRAINING_CONFIG["bf16"],
        eval_strategy=TRAINING_CONFIG["evaluation_strategy"],
        save_strategy=TRAINING_CONFIG["save_strategy"],
        load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
        metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
        greater_is_better=TRAINING_CONFIG["greater_is_better"],
        predict_with_generate=TRAINING_CONFIG["predict_with_generate"],
        generation_max_length=TRAINING_CONFIG["generation_max_length"],
        logging_steps=TRAINING_CONFIG["logging_steps"],
        dataloader_num_workers=TRAINING_CONFIG["dataloader_num_workers"],
        gradient_checkpointing=TRAINING_CONFIG["gradient_checkpointing"],
        remove_unused_columns=False,
        label_names=["labels"],
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=dev_dataset,
        data_collator=train_collator, compute_metrics=compute_metrics,
        processing_class=processor.feature_extractor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=TRAINING_CONFIG["early_stopping_patience"])],
    )

    trainer.train(resume_from_checkpoint=_auto_resume(output_dir))
    del trainer, model, processor
    gc.collect(); torch.cuda.empty_cache()
    print(f"Done: {tag}")

print("Model & train functions defined.")

## 5. Inference Functions

In [ ]:
def get_best_checkpoint(condition_dir):
    """Get best checkpoint from trainer_state of the latest checkpoint dir."""
    ckpts = sorted(
        [d for d in os.listdir(condition_dir) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[-1])
    )
    state_path = os.path.join(condition_dir, ckpts[-1], "trainer_state.json")
    with open(state_path) as f:
        state = json.load(f)
    best = state["best_model_checkpoint"]
    print(f"  Best checkpoint: {os.path.basename(best)} (WER: {state['best_metric']}%)")
    return best

def _run_inference(model, processor, samples, split_name="val"):
    model.eval()
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    hypotheses, references, filenames = [], [], []
    for i in tqdm(range(0, len(samples), BATCH_SIZE), desc=f"Inference [{split_name}]"):
        batch  = samples[i : i + BATCH_SIZE]
        audios = [load_audio(s["audio_path"]) for s in batch]
        inputs = processor.feature_extractor(
            audios, sampling_rate=SAMPLING_RATE,
            return_tensors="pt", padding="max_length", truncation=True,
        ).to(device)
        inputs["input_features"] = inputs["input_features"].to(dtype)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=225)
        preds = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
        hypotheses.extend(preds)
        references.extend([s["text"] for s in batch])
        filenames.extend([os.path.basename(s["audio_path"]) for s in batch])
    return filenames, references, hypotheses

def evaluate_baseline(base_model_id, samples, split_name="val"):
    processor = WhisperProcessor.from_pretrained(base_model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        base_model_id, torch_dtype=torch.float16, device_map="auto", use_safetensors=False,
    )
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="vi", task="transcribe")
    model.eval()
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    hypotheses, references, filenames = [], [], []
    for i in tqdm(range(0, len(samples), BATCH_SIZE), desc=f"Baseline [{split_name}]"):
        batch  = samples[i : i + BATCH_SIZE]
        audios = [load_audio(s["audio_path"]) for s in batch]
        inputs = processor.feature_extractor(
            audios, sampling_rate=SAMPLING_RATE,
            return_tensors="pt", padding="max_length", truncation=True,
        ).to(device)
        inputs["input_features"] = inputs["input_features"].to(dtype)
        with torch.no_grad():
            ids = model.generate(**inputs, forced_decoder_ids=forced_decoder_ids, max_new_tokens=225)
        preds = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
        hypotheses.extend(preds)
        references.extend([s["text"] for s in batch])
        filenames.extend([os.path.basename(s["audio_path"]) for s in batch])
    metrics = compute_all_metrics(references, hypotheses)
    del model, processor; gc.collect(); torch.cuda.empty_cache()
    return filenames, references, hypotheses, metrics

def evaluate_checkpoint(condition_dir, base_model_id, samples, split_name="val"):
    best_ckpt = get_best_checkpoint(condition_dir)
    processor = WhisperProcessor.from_pretrained(base_model_id)
    base = WhisperForConditionalGeneration.from_pretrained(
        base_model_id, torch_dtype=torch.float16, device_map="auto", use_safetensors=False,
    )
    model = PeftModel.from_pretrained(base, best_ckpt)
    filenames, references, hypotheses = _run_inference(model, processor, samples, split_name)
    metrics = compute_all_metrics(references, hypotheses)
    del model, base, processor; gc.collect(); torch.cuda.empty_cache()
    return filenames, references, hypotheses, metrics

def save_hypotheses(filenames, references, hypotheses, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame({"filename": filenames, "reference": references, "hypothesis": hypotheses}
    ).to_csv(path, index=False, encoding="utf-8-sig")

def recompute_norm(hyp_path, normalize_fn=normalize_text):
    """Re-compute norm metrics from CSV without re-running inference."""
    df = pd.read_csv(hyp_path, encoding="utf-8-sig")
    refs_n = [normalize_fn(r) for r in df["reference"]]
    hyps_n = [normalize_fn(h) for h in df["hypothesis"]]
    return {
        "wer_norm": round(wer_metric.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "cer_norm": round(cer_metric.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "ser_norm": round(compute_ser(refs_n, hyps_n), 2),
    }

print("Inference functions defined.")

## 6. Load Data

In [ ]:
print("Loading samples...")
train_samples = load_samples(TRAIN_AUDIO_DIR, TRAIN_TRANSCRIPT)
dev_samples   = load_samples(DEV_AUDIO_DIR,   DEV_TRANSCRIPT)
test_samples  = load_samples(TEST_AUDIO_DIR,  TEST_TRANSCRIPT)
print(f"Train: {len(train_samples)} | Dev: {len(dev_samples)} | Test: {len(test_samples)}")

## 7. Build Dataset Cache

In [ ]:
for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\nBuilding cache: {model_name}")
    model_id  = cfg["model_id"]
    cache_dir = cfg["cache_dir"]
    os.makedirs(cache_dir, exist_ok=True)
    processor = WhisperProcessor.from_pretrained(model_id)

    cache_plain = os.path.join(cache_dir, "plain")
    if os.path.exists(cache_plain):
        print("  [SKIP] plain")
    else:
        ds = prepare_hf_dataset(train_samples, processor, cfg["language"], cfg["task"])
        ds.save_to_disk(cache_plain, writer_batch_size=100)
        print(f"  plain: {len(ds)}")

    cache_aug = os.path.join(cache_dir, "aug")
    if os.path.exists(cache_aug):
        print("  [SKIP] aug")
    else:
        aug_samples = apply_speed_perturbation(train_samples)
        ds = prepare_hf_dataset_aug(aug_samples, processor, cfg["language"], cfg["task"])
        ds.save_to_disk(cache_aug, writer_batch_size=100)
        print(f"  aug: {len(ds)}")

    cache_dev = os.path.join(cache_dir, "dev")
    if os.path.exists(cache_dev):
        print("  [SKIP] dev")
    else:
        ds = prepare_hf_dataset(dev_samples, processor, cfg["language"], cfg["task"])
        ds.save_to_disk(cache_dev, writer_batch_size=100)
        print(f"  dev: {len(ds)}")

    del processor
print("\nAll caches ready.")

## 8. Run Ablation (Val Set)

In [ ]:
ABLATION_CONDITIONS = [
    ("whisper-large-v3", "Baseline",     False, False),
    ("whisper-large-v3", "+ Aug only",   True,  False),
    ("whisper-large-v3", "+ LoRA only",  False, True),
    ("whisper-large-v3", "+ Aug + LoRA", True,  True),
    ("PhoWhisper-large", "Baseline",     False, False),
    ("PhoWhisper-large", "+ Aug only",   True,  False),
    ("PhoWhisper-large", "+ LoRA only",  False, True),
    ("PhoWhisper-large", "+ Aug + LoRA", True,  True),
]

METRICS_DIR = "./output/ablation/metrics"
HYP_DIR     = "./output/ablation/hypotheses"
os.makedirs(METRICS_DIR, exist_ok=True)

all_results = []

for model_name, condition, use_aug, use_lora in ABLATION_CONDITIONS:
    cfg        = MODEL_CONFIGS[model_name]
    model_id   = cfg["model_id"]
    cache_dir  = cfg["cache_dir"]
    output_dir = cfg["output_dir"]
    tag        = f"{model_name}_{condition.replace(' ', '_').replace('+', 'plus')}"

    metrics_path = os.path.join(METRICS_DIR, f"{tag}.json")
    hyp_path     = os.path.join(HYP_DIR,     f"{tag}.csv")

    if os.path.exists(metrics_path):
        print(f"[SKIP] {model_name} | {condition}")
        with open(metrics_path) as f: metrics = json.load(f)
        all_results.append({"model": model_name, "condition": condition, **metrics})
        continue

    print(f"\n{'='*55}")
    print(f"  {model_name} | {condition}")
    print(f"{'='*55}")

    if not use_lora and not use_aug:
        # Baseline zero-shot
        filenames, references, hypotheses, metrics = evaluate_baseline(model_id, dev_samples)

    else:
        # Train condition
        train_cache = "aug" if use_aug else "plain"
        train_ds    = load_from_disk(os.path.join(cache_dir, train_cache))
        dev_ds      = load_from_disk(os.path.join(cache_dir, "dev"))
        ckpt_dir    = os.path.join(output_dir, tag)

        train_condition(
            tag, model_id, train_ds, dev_ds,
            cfg["language"], cfg["task"],
            ckpt_dir, use_spec_augment=use_aug,
        )
        filenames, references, hypotheses, metrics = evaluate_checkpoint(
            ckpt_dir, model_id, dev_samples
        )

    # Save
    save_hypotheses(filenames, references, hypotheses, hyp_path)
    with open(metrics_path, "w") as f: json.dump(metrics, f, indent=2)
    print(f"  WER: {metrics['wer_raw']}% | CER: {metrics['cer_raw']}% | SER: {metrics['ser_raw']}%")
    all_results.append({"model": model_name, "condition": condition, **metrics})

# ── Results table ──────────────────────────────────────────────
df_results = pd.DataFrame(all_results)
cols = ["model", "condition", "wer_raw", "cer_raw", "ser_raw", "wer_norm", "cer_norm", "ser_norm"]
df_results = df_results[cols]
print("\n===== ABLATION RESULTS (VAL SET — BEST CHECKPOINT) =====")
print(df_results.to_string(index=False))
df_results.to_csv("ablation_val_results.csv", index=False, encoding="utf-8-sig")
print("\nSaved: ablation_val_results.csv")

## 9. Holdout Evaluation
⚠️ **Run exactly once with the best model selected from val set.**

In [ ]:
BEST_CKPT     = "./output/ablation/PhoWhisper-large/PhoWhisper-large_aug_lora/checkpoint-1274"
BEST_MODEL_ID = "vinai/PhoWhisper-large"

print("Running holdout evaluation (1 time only)...")
processor = WhisperProcessor.from_pretrained(BEST_MODEL_ID)
base = WhisperForConditionalGeneration.from_pretrained(
    BEST_MODEL_ID, torch_dtype=torch.float16, device_map="auto", use_safetensors=False,
)
model = PeftModel.from_pretrained(base, BEST_CKPT)

filenames, references, hypotheses = _run_inference(model, processor, test_samples, split_name="holdout")
holdout_metrics = compute_all_metrics(references, hypotheses)

del model, base, processor; gc.collect(); torch.cuda.empty_cache()

print("\n===== HOLDOUT RESULTS (TEST SET) =====")
for k, v in holdout_metrics.items():
    print(f"  {k}: {v}")

with open("holdout_results.json", "w") as f:
    json.dump(holdout_metrics, f, indent=2)

save_hypotheses(filenames, references, hypotheses, "holdout_hypotheses.csv")
print("\nSaved: holdout_results.json + holdout_hypotheses.csv")

## 10. Error Analysis (Optional)

In [ ]:
df_hyp = pd.read_csv("holdout_hypotheses.csv", encoding="utf-8-sig")

df_hyp["wer_norm_sample"] = df_hyp.apply(
    lambda row: wer_metric.compute(
        references=[normalize_text(row["reference"])],
        predictions=[normalize_text(row["hypothesis"])]
    ) * 100,
    axis=1
)

df_hyp_sorted = df_hyp.sort_values("wer_norm_sample", ascending=False)
df_hyp_sorted.to_csv("holdout_hypotheses.csv", index=False, encoding="utf-8-sig")
print("Worst predictions (top 20):")
df_hyp_sorted.head(20)

## 11. Re-compute Norm Metrics (if normalize rule changes)

In [ ]:
# Update normalize_text then run this cell — no need to re-run inference
norm_results = []
for model_name, condition, _, _ in ABLATION_CONDITIONS:
    tag      = f"{model_name}_{condition.replace(' ', '_').replace('+', 'plus')}"
    hyp_path = os.path.join(HYP_DIR, f"{tag}.csv")
    if not os.path.exists(hyp_path):
        print(f"[MISSING] {tag}")
        continue
    new_norm = recompute_norm(hyp_path)
    norm_results.append({"model": model_name, "condition": condition, **new_norm})

df_norm = pd.DataFrame(norm_results)
print(df_norm.to_string(index=False))